# [실습] LangChain Tool Call

LLM은 언어 모델이지만, 특수한 형식의 출력을 통해 외부 도구와 소통할 수 있습니다.   

1. LLM에게 프롬프트를 통해 도구의 사용법을 전달합니다.   
2. LLM은 도구가 필요한 경우 파싱 가능한 출력을 생성합니다.   
3. 해당 출력을 파싱하여 실제 도구를 실행하는 과정을 자동화합니다.
4. 실행 결과를 다시 LLM에 전달하면, LLM은 결과를 참고하여 다음 출력을 생성합니다.


# 라이브러리 설정 및 LLM 불러오기

In [1]:
!pip install langchain==0.3.27 langchain_community==0.3.27 langgraph==0.6.8 langchain_google_genai langchain_openai langchain_community langchain_tavily -q

In [2]:
import os
from dotenv import load_dotenv
# OPENAI_API_KEY, GOOGLE_API_KEY
load_dotenv(override=True)

True

In [3]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# Gemini: 무료 API 사용량 존재
# 안정적 서빙을 위해 분당 10개 설정
# 즉, 초당 약 0.167개 요청 (10/60)
# `https://aistudio.google.com/`에서 모델별 사용량 확인

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.167,  # 분당 10개 요청
    check_every_n_seconds=0.1,  # 100ms마다 체크
    max_bucket_size=10,  # 최대 버스트 크기
)


# LLM 초기화
try:
    llm = ChatOpenAI(model='gpt-5-mini', temperature=0.3)
    llm_nonreasoning = ChatOpenAI(model='gpt-4.1-mini', temperature=0.3)
    print("✅ GPT API 사용 가능!")
except:
    print("❌ GPT API 사용 불가- API 키를 확인하세요!")
try:
    llm_gemini = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.7, 
                                rate_limiter=rate_limiter)
    llm_gemini_nonreasoning = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0.3, 
                                rate_limiter=rate_limiter)
    print("✅ Gemini API 사용 가능!")
except:
    print("❌ Gemini API 사용 불가- API 키를 확인하세요!")

✅ GPT API 사용 가능!
✅ Gemini API 사용 가능!


In [4]:
from langchain.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

print("필수 모듈 임포트 완료")

필수 모듈 임포트 완료


## LLM에게 어려운 질문하기   

LLM에게 학습하지 못한 범위의 질문을 전달하면 정확도가 떨어지거나 잘못된 답변을 생성합니다.   

In [5]:
llm.invoke("케이팝 데몬헌터스가 뭐야?")
# 5-mini: 24.6 까지만 

AIMessage(content='어떤 맥락에서 보셨나요? "케이팝 데몬헌터스(K-pop Demon Hunters)"라는 표현은 여러 가지로 쓰일 수 있어서요. 가능한 의미 몇 가지 짚어볼게요.\n\n- 팬메이드 콘셉트나 팬픽: 아이돌을 악마사냥꾼으로 그린 팬아트·팬픽·웹툰 등.  \n- 게임/플랫폼 콘텐츠: 롭록스·포트나이트·모바일게임 등에서 한정적으로 쓰이는 타이틀일 수 있음.  \n- 뮤직비디오(또는 무대) 콘셉트: 어떤 곡이나 퍼포먼스에서 ‘데몬헌터’ 콘셉트를 쓴 경우.  \n- 유튜브·틱톡 같은 소셜미디어의 시리즈나 챌린지 이름일 수도 있음.\n\n어디에서 보셨는지(링크나 스샷, 플랫폼—유튜브/틱톡/트위터/웹툰 등)를 알려주시면 정확히 찾아보고 설명해드릴게요. 그냥 뜻만 궁금하신 거면 위 중 어느 쪽일 가능성이 큰지도 말해주시면 됩니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 844, 'prompt_tokens': 19, 'total_tokens': 863, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 576, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DXPpowpbnKLuBPrIRU80Ba3SUYUIs', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--019db4e6-85a8-78b0-a271-212ef182

In [6]:
llm_gemini.invoke("케이팝 데몬헌터스가 뭐야?")

AIMessage(content='"케이팝 데몬헌터스(K-Pop Demon Hunters)"는 특정 작품 하나의 이름이라기보다는, **케이팝 아이돌의 화려함과 퍼포먼스에 악마나 초자연적인 존재를 사냥하는 \'데몬 헌터\'의 액션/판타지 요소를 결합한 장르나 콘셉트**를 의미합니다.\n\n즉, 무대 위에서는 빛나는 아이돌이지만, 실제로는 비밀리에 초자연적인 위협으로부터 세상을 지키는 능력을 가진 존재라는 설정이죠.\n\n주로 다음과 같은 형태로 나타납니다:\n\n1.  **웹툰/웹소설:**\n    *   가장 활발하게 볼 수 있는 형태입니다. 많은 K-Pop 기획사들이 소속 아이돌 그룹의 세계관(Universe)을 확장하기 위해 웹툰이나 웹소설을 제작하는데, 이 과정에서 아이돌 멤버들이 판타지 세계의 영웅이나 데몬 헌터로 등장하는 경우가 많습니다.\n    *   **예시:**\n        *   **ENHYPEN(엔하이픈)의 \'DARK MOON: 달의 제단\':** 뱀파이어 소년들과 늑대인간들, 그리고 특별한 능력을 가진 소녀의 이야기를 다룬 판타지 웹툰/웹소설입니다.\n        *   **TXT(투모로우바이투게더)의 \'별을 쫓는 소년들\':** 마법 능력을 가진 아이돌 멤버들이 세상을 구하는 이야기를 다룹니다.\n        *   **BTS(방탄소년단)의 \'7FATES: CHAKHO\':** 한국 전통 설화 속 범 사냥꾼(착호갑사)을 모티브로 한 도시 판타지 웹툰/웹소설입니다.\n\n2.  **게임:**\n    *   간혹 인디 게임이나 팬들이 만든 모드(Mod) 등에서 K-Pop 아이돌의 이미지를 차용하거나, 아예 K-Pop 아이돌을 모티브로 한 데몬 헌터 캐릭터가 등장하는 경우가 있습니다.\n    *   **예시:** \'Project: Bellatores\'와 같이 K-Pop 요소를 넣은 데몬 헌팅 게임 프로젝트가 개발되기도 합니다.\n\n3.  **팬픽션/AU (Alternate Universe):**\n    *   팬덤 내에

Anthropic의 기술 블로그에서는, LLM의 외연을 확장하는 방법으로 검색(Retrieval)과 도구(Tool)을 제안했습니다.   
Retrieval은 이후의 RAG에서 자세히 다루도록 하고, 이번 과정에서는 Tool에 대해 이해해 보겠습니다.

## Tool

Tool은 LLM이 이해하고 사용할 수 있는 도구입니다.   
프롬프트를 통해 설명을 전달하므로, 넓은 의미에서 Context Engineering에 해당합니다.   


In [6]:
# OpenAI API의 도구 생성 기능

from openai import pydantic_function_tool
from pydantic import BaseModel, Field

class web_search(BaseModel):
    """웹 검색을 수행합니다."""
    query: str= Field(description="""검색 키워드""")

web_search_tool = pydantic_function_tool(web_search)

web_search_tool

{'type': 'function',
 'function': {'name': 'web_search',
  'strict': True,
  'parameters': {'description': '웹 검색을 수행합니다.',
   'properties': {'query': {'description': '검색 키워드',
     'title': 'Query',
     'type': 'string'}},
   'required': ['query'],
   'title': 'web_search',
   'type': 'object',
   'additionalProperties': False},
  'description': '웹 검색을 수행합니다.'}}

위 결과를 Tool Schema라고 하며, 해당 내용이 LLM의 프롬프트에 포함되어 툴 사용 정보를 전달합니다.

## LangChain Tool 설정하기

LangChain은 Tool Calling을 쉽게 연동하기 위한 기능을 제공합니다.   


LangChain에서 자체적으로 지원하는 Tool을 사용하거나, RAG의 Retriever를 Tool로 변환하는 것도 가능합니다.     
또한, 함수를 Tool로 변환할 수도 있습니다.

가장 대표적인 built-in 툴인 Tavily Search (http://app.tavily.com/) 를 연결해 보겠습니다.

해당 도메인에 접속하여, 구글 계정으로 회원가입을 진행해 주세요.

이후, API 키를 생성하여 env 파일에 `TAVILY_API_KEY`로 추가합니다.

In [7]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [ ]:
from langchain_tavily import TavilySearch

# 파일 업데이트 후 실행
load_dotenv(override=True)

tavily_search = TavilySearch(
    max_results=5,
    # 더 많은 옵션은 Tavily API 문서 Playground 참고
    )

TavilySearch(max_results=5, api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))

In [11]:
from rich import print as rprint
rprint(tavily_search)

TavilySearch(
    max_results=5,
    api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None)
)

In [10]:
search_docs = tavily_search.invoke("케이팝 데몬 헌터스의 의미")
search_docs

{'query': '케이팝 데몬 헌터스의 의미',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://ko.wikipedia.org/wiki/%EC%BC%80%EC%9D%B4%ED%8C%9D_%EB%8D%B0%EB%AA%AC_%ED%97%8C%ED%84%B0%EC%8A%A4',
   'title': '케이팝 데몬 헌터스 - 위키백과, 우리 모두의 백과사전',
   'content': '케이팝 데몬 헌터스는 한국 문화에 영감을 받은 이야기를 만들고자 했던 강 감독의 바람에서 비롯되었으며, 신화, 악마학, K-pop 요소를 활용하여 시각적으로 독특하고',
   'score': 0.9999151,
   'raw_content': None},
  {'url': 'https://blog.naver.com/mellybird/223978556012?viewType=pc',
   'title': '데몬 뜻, 케데헌에서의 의미 그리고 데몬의 진짜 발음은? - Naver Blog',
   'content': "K-pop의 위상이 갈수록 높아지고 있는 가운데, '케이팝 데몬 헌터스'의 이름에 있는 '데몬'의 뜻과 정확한 발음 그리고 이 애니매이션과 어떤 관련이",
   'score': 0.99990165,
   'raw_content': None},
  {'url': 'https://www.hjn24.com/news/articleView.html?idxno=134214',
   'title': '‘케이팝 데몬 헌터스’에 열광하는 이유 - 홍주일보',
   'content': '음악적 요소만이 아니라 ‘케데헌’에는 남산의 서울타워, 한복, 한글, 한국의 전통무기, 호랑이, 까치, 김밥, 무당 등과 같은 한국적 요소들이 곳곳에 산재(散在)해 있다. 한국의 정신과 문화, 상품이 세계 곳곳에 더욱 진출할 수 있는 계기가 될 수 있다. 그동안 ‘한류’에서 외국 팬덤이 주로 한국

API 호출은 대부분 사용자가 원하는 정보보다 많은 내용을 전달합니다.   
불필요한 내용을 제거하는 함수를 구성합니다.

In [ ]:
from langchain_core.tools import tool

@tool
def tavily_search(query, max_results = 5):
    """Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.
query는 검색어를 의미하며, max_results는 최대 검색 문서 수를 의미합니다.
사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 최대한 검색하라고 하는 경우 20을 사용하세요."""
    tavily_search = TavilySearch(max_results=max_results)
    results = tavily_search.invoke(query)['results']

    context = ''
    for result in results:
        # result: dict
        doc_content = 'URL: '+ result.get('url') +'\n Title:'+ result.get('title') +'\n Content:' + result.get('content')
        context += doc_content + '\n\n---\n\n'

    return context

print(tavily_search.invoke({'query':'케이팝 데몬헌터스'}))





URL: https://ko.wikipedia.org/wiki/%EC%BC%80%EC%9D%B4%ED%8C%9D_%EB%8D%B0%EB%AA%AC_%ED%97%8C%ED%84%B0%EC%8A%A4
 Title:케이팝 데몬 헌터스 - 위키백과, 우리 모두의 백과사전
 Content:《케이팝 데몬 헌터스》(영어: KPop Demon Hunters )는 2025년 공개된 미국의 뮤지컬, 판타지, 코미디 애니메이션 영화이다. 소니 픽처스 애니메이션이 제작하고 넷플릭스

---

URL: https://www.netflix.com/kr/title/81498621
 Title:케이팝 데몬 헌터스, 지금 시청하세요 | 넷플릭스 공식 사이트 - Netflix
 Content:케이팝 슈퍼스타 루미, 미라, 조이. 매진을 기록하는 대형 스타디움 공연이 없을 때면 이들은 또 다른 활동에 나선다. 바로 비밀 능력을 이용해 팬들을 초자연적 위협

---

URL: https://www.youtube.com/playlist?list=PLL1qZqiuec_T3hd2acyx2RNV9g9-wSS8t
 Title:케이팝 데몬 헌터스 KPop Demon Hunters - YouTube
 Content:Home. Shorts. Library. Play all. 케이팝 데몬 헌터스 KPop Demon Hunters. Netflix Korea 넷플릭스 코리아. Playlist•17 videos•184,748 views.

---

URL: https://namu.wiki/w/%EC%BC%80%EC%9D%B4%ED%8C%9D%20%EB%8D%B0%EB%AA%AC%20%ED%97%8C%ED%84%B0%EC%8A%A4
 Title:케이팝 데몬 헌터스 - 나무위키:대문
 Content:+ 이 사운드트랙 호평에 힘입어, 2025년 7월 4일에는 미국 스포티파이 데일리 송 차트에서 사자 보이즈의 Your Idol(**1위**)과, Soda Pop(10위)이, 그리고 헌트릭스의 Golden "Golden(케이팝

위에서 만든 함수에, @tool decorator를 붙이면 랭체인의 툴로 변환됩니다.

## LLM에 Tool 연결하기   

생성한 툴은 llm.bind_tools()를 통해 LLM에 연결할 수 있습니다.    

In [13]:
tools = [tavily_search]

llm_with_tools = llm.bind_tools(tools)
rprint(llm_with_tools)

RunnableBinding(
    bound=ChatOpenAI(
        client=<openai.resources.chat.completions.completions.Completions object at 0x0000024E28F46120>,
        async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000024E28F46BA0>,
        root_client=<openai.OpenAI object at 0x0000024E274F5940>,
        root_async_client=<openai.AsyncOpenAI object at 0x0000024E28F46900>,
        model_name='gpt-5-mini',
        model_kwargs={},
        openai_api_key=SecretStr('**********'),
        openai_proxy=None,
        stream_usage=True
    ),
    kwargs={
        'tools': [
            {
                'type': 'function',
                'function': {
                    'name': 'tavily_search',
                    'description': 'Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.\nquery는 검색어를 의미하며, 
max_results는 최대 검색 문서 수를 의미합니다.\n사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 
최대한 검색하라고 하는 경우 20을 사용하세요.',
                    'parameters': {
                        'properties': {'query': {}, 'max_results': {'default': 5}},
                        'required': ['query'],
                        'type': 'object'
                    }
                }
            }
        ]
    },
    config={},
    config_factories=[]
)

랭체인에서, tool 정보는 tools에 저장되는데요.   
해당 내용은 랭체인 내부에서 json Schema 형식으로 프롬프트에 포함됩니다.

In [14]:
llm_with_tools.kwargs['tools']

[{'type': 'function',
  'function': {'name': 'tavily_search',
   'description': 'Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.\nquery는 검색어를 의미하며, max_results는 최대 검색 문서 수를 의미합니다.\n사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 최대한 검색하라고 하는 경우 20을 사용하세요.',
   'parameters': {'properties': {'query': {}, 'max_results': {'default': 5}},
    'required': ['query'],
    'type': 'object'}}}]

LLM은 프롬프트로 주어지는 툴 정보를 바탕으로 Tool의 사용을 결정합니다.    
Schema를 통해, 툴의 의미와 사용 방법, 형식을 이해합니다.

Tool을 탑재한 LLM을 실행해 보겠습니다.

In [24]:
from langchain_core.output_parsers import StrOutputParser

In [29]:
prompt = ChatPromptTemplate(
    [
        ('system', '주어진 검색 툴을 사용해서 질문에 답변하세요.'),
        ('human', '{query}')
    ]
)
# parser = StrOutputParser()
chain = prompt | llm_with_tools
# chain = prompt | llm | parser
# tool_call_msg = chain.invoke('삼성SDS의 GPUaaS 서비스가 뭐야?')
tool_call_msg = llm_with_tools.invoke('삼성SDS의 GPUaaS 서비스가 뭐야?')

rprint(tool_call_msg)

AIMessage(
    content='',
    additional_kwargs={
        'tool_calls': [
            {
                'id': 'call_ac0seiawysSExLmAgTnePeda',
                'function': {
                    'arguments': '{"query":"삼성SDS GPUaaS GPU as a Service 삼성SDS 서비스 설명"}',
                    'name': 'tavily_search'
                },
                'type': 'function'
            }
        ],
        'refusal': None
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 104,
            'prompt_tokens': 208,
            'total_tokens': 312,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 64,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
        },
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-DXSMCxCfsZJqeDlqiEPzgH7Hfrp7M',
        'service_tier': 'default',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='run--019db57a-8a29-7620-ac58-91fc5a73967d-0',
    tool_calls=[
        {
            'name': 'tavily_search',
            'args': {'query': '삼성SDS GPUaaS GPU as a Service 삼성SDS 서비스 설명'},
            'id': 'call_ac0seiawysSExLmAgTnePeda',
            'type': 'tool_call'
        }
    ],
    usage_metadata={
        'input_tokens': 208,
        'output_tokens': 104,
        'total_tokens': 312,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 64}
    }
)

In [26]:
tool_call_msg

''

In [34]:
tool_list = {'tavily_search': tavily_search}
tool_name = tool_call_msg.tool_calls[0]['name']
tool_exec = tool_list[tool_name]
tool_msg = tool_exec.invoke(tool_call_msg.tool_calls[0])

messages = []
messages.append(HumanMessage('삼성SDS의 GPUaaS 서비스가 뭐야?'))
messages.append(tool_call_msg)
messages.append(tool_msg)
chain = llm_with_tools | StrOutputParser()
result = chain.invoke(messages)


In [36]:
print(result)

간단히 말하면, 삼성SDS의 GPUaaS(GPU as a Service)는 고성능 GPU 자원을 클라우드 형태로 빌려 쓰는 서비스입니다. 기업이나 개발자가 비싼 GPU를 직접 구매·운영하지 않아도, 필요할 때 바로 고성능 연산 환경을 쓸 수 있게 해주며 AI 모델 실험·학습·배포 전 과정에 맞춘 환경과 운영 지원을 제공합니다.

주요 내용(요약)
- 무엇을 제공하나: 엔비디아 H100·B300(Blackwell 계열) 등 고성능 GPU 기반 인프라를 온디맨드 또는 구독형으로 제공. 베어메탈 GPU 클러스터도 지원.
- 대상 작업: AI 모델 실험/개발, 대규모 분산 학습(대형 LLM 학습), 추론(서빙), 고성능 컴퓨팅(HPC) 등.
- 장점: 초기 투자 비용 절감(구매 불필요), 즉시 사용 가능한 환경, 대규모 분산 처리에 최적화된 구성, MLOps·AI 서비스와 연계, 24/7 장애 대응 및 전문 기술 지원, 기업용 보안·운영 체계.
- 과금 방식: 구독형(또는 사용량 기반)으로 필요한 만큼만 비용 지불(자세한 요금은 상담 필요).
- 추가 기능/계획: 서버리스 추론, 자동 분산 학습 같은 서비스 연계 기능을 제공하거나 계획 중이라는 안내가 있음.

누가 쓰면 좋은가
- AI 모델을 개발·배포하는 기업(스타트업 ~ 대기업)
- 대규모 학습 자원이 필요한 연구팀
- 자체 인프라 관리 부담을 줄이고 싶은 개발팀

원하면 삼성SDS 공식 페이지(서비스 설명)와 현재 제공되는 GPU 종류/사양, 요금제, 도입 절차를 찾아 자세히 보여드리거나, 귀하의 사용 사례에 맞는 추천 구성을 함께 제안해 드릴게요. 어떤 정보가 필요하세요?


In [49]:
prompt = ChatPromptTemplate(
    [
        ('system', '주어진 검색 툴을 사용해서 질문에 답변하세요.'),
        ('human', '{query}'),
        ('ai', '{tool_call_msg}'),
        ('ai', '{tool_msg}')
    ]
)

chain = prompt | llm_with_tools | parser

result = chain.invoke({'query': '삼성SDS의 GPUaaS 서비스가 뭐야?', 'tool_call_msg' : tool_call_msg, 'tool_msg' : tool_msg})

In [50]:
print(result)

간단히 말하면, 삼성SDS의 GPUaaS(GPU as a Service)는 복잡한 인프라 구축 없이 고성능 GPU 자원을 필요할 때마다 빌려 쓰도록 제공하는 관리형 서비스입니다. 요약하면 다음과 같습니다.

- 무엇인지  
  - 삼성 클라우드 플랫폼(SCP)에서 제공하는 온디맨드 GPU 구독(임대) 서비스로, AI 학습·추론·서빙용 고성능 GPU를 즉시 사용할 수 있게 해 줍니다.  

- 핵심 기능·특징  
  - 고성능 GPU(예: 엔비디아 B300/Blackwell, H100 등)를 클라우드에서 제공  
  - 베어메탈 GPU 클러스터 등 대규모 학습에 적합한 환경 지원  
  - 24/7 장애 대응·안정적 운영과 전문 기술 지원 제공  
  - MLOps 및 삼성SDS의 AI/ML 서비스들과 연계 가능  
  - 구독형/사용량 기반 과금으로 초기 투자 부담 감소  
  - (공식·언론 보도 기준) 앞으로 서버리스 추론·자동 분산 학습 같은 서비스도 순차적으로 제공 예정

- 적합한 사용 사례  
  - 모델 실험·개발(빠른 프로토타이핑)  
  - LLM·대규모 모델 학습 및 분산 학습  
  - 고성능 추론(서비스화) 및 배포

- 장점  
  - GPU 구매·운영 부담(초기비용·관리) 감소  
  - 최신 GPU를 바로 사용 가능 → 학습 속도·처리량 향상  
  - 기업 데이터가 클라우드 내에서 관리되어 보안 우려 완화

- 시작 방법(일반 절차)  
  - 삼성SDS GPUaaS 안내 페이지 확인 → 상담 신청 또는 계정/콘솔 통해 자원 신청  
  - 필요 사양(모델 크기·분산 여부)에 맞춰 인스턴스·클러스터 선택 후 사용

참고(공식 페이지/기사)  
- 삼성SDS GPUaaS 소개: https://www.samsungsds.com/kr/gpuaas/gpuaas.html  
- 관련 보도: “삼성SDS, 엔비디아 최신 GPU 구독 서비스 출시” (조선일보 등)

원하시면 귀사(또는 개인) 용도에 맞는 권장 구성(예: 연구용·서비스용·대규모 

`tool_calls`에 포함된 name을 활용하면 툴 결과를 전달할 수 있습니다.   
`name` 값은 문자열이므로, dictionary를 통해 연결합니다.

In [22]:
tool_call_msg.tool_calls

[{'name': 'tavily_search',
  'args': {'query': '삼성SDS GPUaaS 서비스', 'max_results': 5},
  'id': 'call_3Wm6eG64P7lhGPPT1oeigABQ',
  'type': 'tool_call'}]

Tool에 tool_call을 입력해 invoke를 수행합니다.

In [16]:
tool_list = {'tavily_search': tavily_search}
tool_exec = tool_list[tool_call_msg.tool_calls[0]['name']]
tool_exec

StructuredTool(name='tavily_search', description='Tavily 검색을 통해 인터넷 검색 결과를 가져옵니다.\nquery는 검색어를 의미하며, max_results는 최대 검색 문서 수를 의미합니다.\n사용자가 별도의 요청을 하지 않으면 max_results는 5를 사용하고, 최대한 검색하라고 하는 경우 20을 사용하세요.', args_schema=<class 'langchain_core.utils.pydantic.tavily_search'>, func=<function tavily_search at 0x0000024E29A36770>)

In [17]:
tool_msg = tool_exec.invoke(tool_call_msg.tool_calls[0])
print(tool_msg.content)

URL: https://www.samsungsds.com/kr/event/gpuaas-event1.html
 Title:GPU as a Service GPUaaS 고민, 삼성SDS가 모두 해결해 ...
 Content:삼성SDS GPUaaS가 좋은 이유 · 1. NVIDIA H100 등 국내 최대 GPU 보유량 기반, 국내 최대 규모 GPU 서비스 운영 · 2. 도입 상담부터 구축, 원활한 운영까지 컨설턴트의 End-

---

URL: https://www.samsungsds.com/kr/insights/gpuaas.html
 Title:AI시대, GPU를 서비스하다(GPUaaS) | 인사이트리포트
 Content:GPUaaS는 AI 시대에 필수적인 클라우드 기반 GPU 서비스로, 온프레미스 대비 비용 효율성과 유연성을 제공합니다. · 머신러닝, 클라우드 게임, 3D 렌더링,

---

URL: https://www.mk.co.kr/news/it/11995766
 Title:“엔비디아 GPU 구독하세요”...삼성SDS, 국내 최초 서비스 출시
 Content:삼성SDS의 GPUaaS 서비스는 국내 기업들의 AI 기술 도입 장벽을 낮추고, 정부가 추진하는 AI 클라우드 서비스 확충 노력과도 맥을 같이 한다고 볼

---

URL: https://www.youtube.com/watch?v=6qKQ49BjiJ8
 Title:[Samsung SDS] Samsung Cloud Platform GPUaaS, a ...
 Content:Samsung Cloud Platform의 GPU as a Service는 경쟁사 대비 신속한 기술 지원과 안정적인 인프라 운영이라는 차별적 강점을 제공합니다.

---

URL: https://www.samsungsds.com/kr/gpuaas/gpuaas.html
 Title:삼성 클라우드 플랫폼 GPUaaS | 즉시 사용 가능한 고성능 GPU 도입 | 삼성SDS
 Content:### 삼성 클라우드 플랫폼 GPU as a Servic

ToolMessage라는 새로운 형태의 메시지가 생성되었습니다.

HumanMessage, AIMessage(Tool Call), ToolMessage를 모두 전달합니다.

In [18]:
tool_msg

ToolMessage(content='URL: https://www.samsungsds.com/kr/event/gpuaas-event1.html\n Title:GPU as a Service GPUaaS 고민, 삼성SDS가 모두 해결해 ...\n Content:삼성SDS GPUaaS가 좋은 이유 · 1. NVIDIA H100 등 국내 최대 GPU 보유량 기반, 국내 최대 규모 GPU 서비스 운영 · 2. 도입 상담부터 구축, 원활한 운영까지 컨설턴트의 End-\n\n---\n\nURL: https://www.samsungsds.com/kr/insights/gpuaas.html\n Title:AI시대, GPU를 서비스하다(GPUaaS) | 인사이트리포트\n Content:GPUaaS는 AI 시대에 필수적인 클라우드 기반 GPU 서비스로, 온프레미스 대비 비용 효율성과 유연성을 제공합니다. · 머신러닝, 클라우드 게임, 3D 렌더링,\n\n---\n\nURL: https://www.mk.co.kr/news/it/11995766\n Title:“엔비디아 GPU 구독하세요”...삼성SDS, 국내 최초 서비스 출시\n Content:삼성SDS의 GPUaaS 서비스는 국내 기업들의 AI 기술 도입 장벽을 낮추고, 정부가 추진하는 AI 클라우드 서비스 확충 노력과도 맥을 같이 한다고 볼\n\n---\n\nURL: https://www.youtube.com/watch?v=6qKQ49BjiJ8\n Title:[Samsung SDS] Samsung Cloud Platform GPUaaS, a ...\n Content:Samsung Cloud Platform의 GPU as a Service는 경쟁사 대비 신속한 기술 지원과 안정적인 인프라 운영이라는 차별적 강점을 제공합니다.\n\n---\n\nURL: https://www.samsungsds.com/kr/gpuaas/gpuaas.html\n Title:삼성 클라우드 플랫폼 GPUaaS | 즉시 사용 가능한 고성능 GPU 도입 | 

In [19]:
messages= []

messages.append(HumanMessage('삼성SDS의 GPUaaS 서비스가 뭐야?'))
# 질문이 들어있는 HumanMessage + 툴 요청이 들어있는 AIMessage + 툴 결과가 들어있는 ToolMessage
messages.append(tool_call_msg)
messages.append(tool_msg)



`Query-Tool Call-Tool`의 형식은 가장 기본적인 툴 사용 방법입니다.

In [20]:
result = llm_with_tools.invoke(messages)
print(result.content)

간단히 말하면, 삼성SDS의 GPUaaS는 "GPU를 서비스로 제공"하는 클라우드 상품입니다. AI 모델 학습·추론, 대규모 연산(분산 학습), 3D 렌더링 등 GPU가 필요한 워크로드를 위해 하드웨어 관리 없이 즉시 고성능 GPU 자원을 빌려 쓰도록 해줍니다.

주요 내용(한눈에)
- 무엇인가: 클라우드 기반 GPU 인프라(온디맨드·구독형) + 운영·기술지원. 삼성 Cloud Platform에서 제공.
- 제공 자원: NVIDIA H100 등 최신 고성능 GPU를 포함한 대규모 GPU 풀(국내 최대 규모 보유량을 표방).
- 형태: 베어메탈 GPU 클러스터(대규모 학습용)와 즉시 사용 가능한 인스턴스 형태 등 목적에 맞는 옵션 제공.
- 기능·지원: MLOps·AI/ML 서비스 연계, 컨설팅·구축·운영 지원, 24/7 장애대응 등 엔터프라이즈 수준 운영 체계.
- 적합한 사용처: 딥러닝 모델 연구·개발, 대규모 분산 학습(LLM 등), 실시간 추론 서비스, 고해상도 렌더링, GPU 가속 테스트 등.

장점
- 하드웨어 구매·운영 부담 감소(초기 투자 절감)
- 빠른 시작과 확장성(필요할 때만 자원 사용)
- 전문 운영·지원으로 안정성 확보
- 국내 데이터·규제 요건을 고려한 서비스(국내 리전·운영 기반)

주의할 점
- 비용은 사용량·GPU 종류(예: H100)·구성에 따라 달라짐 — 정확한 견적은 상담 필요
- 네트워크·데이터 전송 비용, 데이터 보안·거버넌스 요구사항은 사전 확인 권장
- 일부 고성능 대규모 워크로드는 네트워크·스토리지 구성 고려 필요

원하면 다음을 도와드릴게요
- 귀사(혹은 개인)의 사용 사례에 맞는 GPU 사양·요금 예측
- 삼성SDS GPUaaS와 다른 클라우드(GCP/AWS/Azure) 비교
- 도입 절차·보안·데이터 관리 체크리스트 정리

공식 소개 페이지(참고): 삼성SDS GPUaaS 서비스 페이지를 확인하면 상세 기능·옵션을 볼 수 있습니다. 원하시면 제가 주요 링크(요금·사양 페이지)를 찾아 정리해 드릴게요.


In [21]:
result = chain.invoke(messages)
print(result.content)

간단히 말하면, 삼성SDS의 GPUaaS는 “GPU를 서비스 형태로 제공”하는 클라우드 상품으로, 복잡한 인프라 관리 없이 고성능 GPU 자원을 바로 쓰면서 AI 모델 개발·학습·추론을 빠르게 진행할 수 있게 해주는 서비스입니다.

주요 내용(요약)
- 서비스 개념: 클라우드 기반 GPU 리소스(임대형) + 운영·지원 서비스를 묶은 형태. 인프라 설치·관리 없이 GPU 연산을 즉시 사용 가능.
- 대상 워크로드: 머신러닝/딥러닝 학습 및 실험, 대규모 모델(LLM) 학습, 추론 서빙, 3D 렌더링, 클라우드 게임 등 GPU 연산이 필요한 작업.
- 제공 인프라: NVIDIA H100 등 최신 고성능 GPU 제공(삼성SDS는 국내 최대 수준의 GPU 보유·운영을 내세움). 베어메탈 GPU 클러스터(대규모 학습용) 등 옵션도 있음.
- 운영·지원: 도입 상담 → 구축 → 운영까지 컨설팅·기술지원 제공. 24시간 장애 대응 및 운영 노하우 기반 관리.
- 제품/포지셔닝: 삼성 클라우드 플랫폼 기반의 GPUaaS / B300 GPUaaS 등(즉시 사용 가능한 고성능 GPU, MLOps·AI 서비스 연계 강조).
- 장점: 초기 투자·인프라 복잡성 감소, 필요할 때만 확장·사용 가능, 전문 운영 지원으로 안정성 확보, 기업용(온프레 대비 비용/유연성 이점).
- 차별점(삼성SDS 주장): 국내 최대 규모 GPU 운용·보유, 신속한 기술 지원, 엔드투엔드 컨설팅.

실제 사용 흐름(예시)
1. 상담으로 워크로드·성능 요구 파악
2. 적합한 GPU 유형(예: H100 기반 베어메탈/가상머신 등)과 요금 플랜 결정
3. 계정/리소스 할당 후 즉시 개발·학습 시작
4. 필요 시 스케일업·스케일아웃, 운영/모니터링/장애대응 지원

원하시면 더 자세히 알려드릴게요:
- 가격·과금 방식(시간당/구독 등) 조회
- 제공되는 GPU 종류/사양과 선택 가이드
- SLA(가용성), 보안·네트워크 옵션
- 삼성SDS 공식 페이지 링크 및 연락처

어떤 정보가 더 필요하세요? (예: 가격 비

In [22]:
result = llm.invoke(messages)
print(result.content)

간단히 말하면, 삼성SDS의 GPUaaS(GPU as a Service)는 기업들이 직접 GPU 인프라를 사서 운영하지 않아도 클라우드 형태로 고성능 GPU를 즉시 사용할 수 있게 해 주는 서비스입니다. AI 모델 학습·추론, 대규모 분산 학습, 3D 렌더링 등 GPU 연산이 필요한 워크로드를 대상으로 합니다.

주요 특징
- 고성능 GPU 제공: NVIDIA H100 등 최신 GPU를 포함해 대규모 GPU 풀을 보유(삼성SDS 발표 기준).  
- 다양한 제공 형태: 베어메탈 GPU 클러스터(대규모 학습용) 등 목적에 맞는 인스턴스/구성 선택 가능.  
- 즉시 사용·관리 편의성: 복잡한 인프라 구축 없이 바로 인스턴스 할당·사용 가능.  
- 운영·지원: 삼성SDS의 운영 노하우와 24/7 장애 대응, 컨설팅·기술지원 제공.  
- 연계 서비스: MLOps, 데이터플랫폼, 클라우드 서비스와 연계해 개발→배포까지 지원.

주요 사용 사례
- 대형 AI 모델(LLM) 학습 및 분산 훈련  
- 추론(서빙)·실시간 응답 서비스  
- 모델 실험(프로토타이핑), 하이퍼파라미터 탐색  
- 그래픽 렌더링·시뮬레이션 등 고성능 컴퓨팅

장점
- 초기 투자·운영 부담 감소(하드웨어 구매·관리 불필요)  
- 필요할 때만 확장·축소 가능해 비용 효율성 확보  
- 전문 지원으로 도입 속도·안정성 향상

주의할 점 / 고려사항
- 비용 구조 확인(시간 단위/구독형/견적 방식 등)  
- 데이터 전송량·보안(민감 데이터의 온프레미스 요건 여부)  
- GPU 리소스 가용성(대규모 예약 시 사전 협의 필요)

시작 방법
1. 사용 목적(학습/추론/렌더링), 성능 요구량(NVIDIA H100 등), 예산을 정리.  
2. 삼성SDS GPUaaS 상담/견적 요청 또는 데모 신청.  
3. 설정·테스트 후 운영·확장.

더 자세한 정보나 견적이 필요하면 삼성SDS 공식 GPUaaS 소개 페이지를 보거나(삼성 클라우드 플랫폼 GPUaaS), 삼성SDS에 직접 문의해 보세요. 필요하면